In [ ]:
library(dplyr)
library(NADA2)

In [ ]:
# Load original data
df <- read.csv("C:/Users/leila/Dropbox/MayoWetlands/wetland_alldata_2025_only.csv")

# Separate upland and wetland sites
upland_sites <- df %>% filter(grepl("upland", SiteID, ignore.case = TRUE))
wetland_sites <- df %>% filter(!grepl("upland", SiteID, ignore.case = TRUE))

cat(sprintf("Upland sites: %d\n", nrow(upland_sites)))
cat(sprintf("Wetland sites: %d\n", nrow(wetland_sites)))

# Define chemistry columns: Al to ORPmV
chem_start <- which(names(wetland_sites) == "Al")
chem_end <- which(names(wetland_sites) == "ORPmV")
chem_cols <- names(wetland_sites)[chem_start:chem_end]

# Remove TDS
if ("TDS_mg.L" %in% names(wetland_sites)) {
  wetland_sites <- wetland_sites %>% select(-TDS_mg.L)
  upland_sites <- upland_sites %>% select(-TDS_mg.L)
}
chem_cols <- chem_cols[chem_cols != "TDS_mg.L"]

cat(sprintf("\nChemistry variables to process: %d\n", length(chem_cols)))

# Calculate censored proportions
censored_proportions <- sapply(wetland_sites[chem_cols], function(x) {
  not_na <- !is.na(x)
  if(sum(not_na) > 0) {
    return(sum(x[not_na] < 0) / sum(not_na))
  }
  return(0)
})

# Set ORPmV censored proportion to 0 (has legitimate negatives)
if("ORPmV" %in% names(censored_proportions)) {
  censored_proportions["ORPmV"] <- 0
}

# Drop highly censored variables
cols_to_drop <- names(censored_proportions[censored_proportions >= 0.60])
cat(sprintf("\nDropping %d variables with ≥60%% censoring\n", length(cols_to_drop)))
if(length(cols_to_drop) > 0) {
  print(cols_to_drop)
  wetland_sites <- wetland_sites %>% select(-all_of(cols_to_drop))
  upland_sites <- upland_sites %>% select(-all_of(cols_to_drop))
}

chem_cols_retained <- names(censored_proportions[censored_proportions < 0.60])



# Custom Function

In [ ]:
# ============================================================
# STEP 1: Calculate u-scores on RAW data (68 obs)
# ============================================================
cat("\n=== STEP 1: CALCULATING U-SCORES ON RAW DATA (68 obs) ===\n")

for(col in chem_cols_retained) {
  values <- wetland_sites[[col]]
  not_na <- !is.na(values)
  
  if(sum(not_na) > 0) {
    values_no_na <- values[not_na]
    n <- length(values_no_na)
    
    if(col == "ORPmV") {
      ranks <- rank(values_no_na, ties.method = "average")
    } else {
      ranks <- rank(abs(values_no_na), ties.method = "average")
    }
    
    uscore_values <- rep(NA, length(values))
    uscore_values[not_na] <- qnorm((ranks - 0.5) / n)
    wetland_sites[[col]] <- uscore_values
  }
}

cat(sprintf("U-scores calculated on %d observations\n", nrow(wetland_sites)))

# Reorder columns - chemistry at END
non_chem_cols <- setdiff(names(wetland_sites), chem_cols_retained)
wetland_sites <- wetland_sites %>% select(all_of(non_chem_cols), all_of(chem_cols_retained))
upland_sites <- upland_sites %>% select(all_of(non_chem_cols), all_of(chem_cols_retained))

# ============================================================
# EXPORT #1: U-SCORES WITHOUT AVERAGING (68 wetland obs)
# ============================================================
df_uscore_68obs <- bind_rows(wetland_sites, upland_sites)

output_path_68 <- "C:/Users/leila/Dropbox/MayoWetlands/wetland_alldata_2025_only_uscore.csv"
write.csv(df_uscore_68obs, output_path_68, row.names = FALSE)

cat(sprintf("\n✓ EXPORTED #1: %s\n", output_path_68))
cat(sprintf("  %d rows (%d wetland + %d upland)\n", 
            nrow(df_uscore_68obs), nrow(wetland_sites), nrow(upland_sites)))
cat("  U-scores calculated, NOT averaged\n")

# [OPTIONAL] Compare the custom function and NADA2 function

In [ ]:
library(NADA2)
library(dplyr)


# Drop upland sites
df <- df %>%
  filter(!grepl("upland", SiteID, ignore.case = TRUE))

# Define chemistry columns
chem_cols <- names(df)[which(names(df) == "Al") : which(names(df) == "SO4_mg.L")]

# Convert "<" to negative values if needed
for(col in chem_cols) {
  if(is.character(df[[col]])) {
    df[[col]] <- ifelse(grepl("<", df[[col]]), 
                        -as.numeric(gsub("<", "", df[[col]])),
                        as.numeric(df[[col]]))
  }
}

# Test NADA2::uscore on Al column
cat("=== TESTING NADA2::uscore() ===\n")
test_vals <- df$Al[!is.na(df$Al)]

cat(sprintf("Total values: %d\n", length(test_vals)))
cat("First 20 values:\n")
print(head(test_vals, 20))

# Prepare for uscore
y_vals <- abs(test_vals)  # absolute values
cen_ind <- test_vals < 0   # censored indicator (TRUE if censored)

cat(sprintf("\nCensored: %d (%.1f%%)\n", sum(cen_ind), sum(cen_ind)/length(cen_ind)*100))

# Call NADA2::uscore with rnk = FALSE
uscores <- NADA2::uscore(y_vals, cen_ind, rnk = FALSE)

cat("\nU-scores from NADA2::uscore(rnk = FALSE):\n")
cat("First 20 u-scores:\n")
print(head(uscores, 20))

cat(sprintf("\nU-score range: %.3f to %.3f\n", min(uscores), max(uscores)))
cat(sprintf("U-score mean: %.3f\n", mean(uscores)))
cat(sprintf("U-score sd: %.3f\n", sd(uscores)))

# Check if they're continuous or integers
cat("\nAre u-scores integers?\n")
cat(sprintf("All integers: %s\n", all(uscores == round(uscores))))

# Also test with rnk = TRUE to compare
uscores_rank <- NADA2::uscore(y_vals, cen_ind, rnk = TRUE)
cat("\n=== COMPARISON: rnk = TRUE ===\n")
cat("First 20 values:\n")
print(head(uscores_rank, 20))
cat(sprintf("Range: %.3f to %.3f\n", min(uscores_rank), max(uscores_rank)))

# NEW Average the u-score per site

In [ ]:
# ============================================================
# STEP 2: Average the u-scores per site
# ============================================================
cat("\n=== STEP 2: AVERAGING U-SCORES PER SITE ===\n")

wetland_sites_avg <- wetland_sites %>%
  mutate(SiteGroup = substr(SiteID, 1, 3)) %>%
  group_by(SiteGroup) %>%
  summarise(
    across(all_of(chem_cols_retained), ~ mean(.x, na.rm = TRUE)),
    across(-all_of(chem_cols_retained), ~ dplyr::first(.x)),
    .groups = "drop"
  ) %>%
  mutate(SiteID = SiteGroup) %>%
  select(-SiteGroup)

cat(sprintf("Averaged to %d sites\n", nrow(wetland_sites_avg)))

# Reorder averaged data
wetland_sites_avg <- wetland_sites_avg %>% 
  select(all_of(non_chem_cols), all_of(chem_cols_retained))

# ============================================================
# EXPORT #2: U-SCORES WITH AVERAGING (27 wetland sites)
# ============================================================
df_uscore_avg <- bind_rows(wetland_sites_avg, upland_sites)

output_path_avg <- "C:/Users/leila/Dropbox/MayoWetlands/wetland_alldata_2025_only_uscore_averaged.csv"
write.csv(df_uscore_avg, output_path_avg, row.names = FALSE)

cat(sprintf("\n✓ EXPORTED #2: %s\n", output_path_avg))
cat(sprintf("  %d rows (%d wetland + %d upland)\n", 
            nrow(df_uscore_avg), nrow(wetland_sites_avg), nrow(upland_sites)))
cat("  U-scores calculated AND averaged\n")